# Variational Mean-Field for the Chiral Spin Model

This notebook illustrates the **quadratic variational mean-field** (VMF)
approximation on a spin-1/2 triangular strip with a scalar chirality term

$$H = J\sum_{\langle ij\rangle} \vec{S}_i\cdot\vec{S}_j
+ K\sum_{\triangle} \vec{S}_i\cdot(\vec{S}_j\times\vec{S}_k)$$

We track two complementary convergence diagnostics as the number of
auxiliary fields $m$ grows:

- **Variational free energy** $F[\sigma_m]$ — upper bound on $F_{\rm exact}$.
- **Variance ratio** $R_m = \mathrm{Var}[\sigma_m]/\mathrm{Var}[\sigma_{\rm SC}]$ — measures reduction in fluctuations relative to the self-consistent baseline.

For small systems ($L\le 8$) we also compute the exact free energy and
**T-score** via full diagonalisation.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from qalma import graph_from_alps_xml, model_from_alps_xml
from qalma.model import SystemDescriptor
from qalma.meanfield import (
    variational_quadratic_mfa,
    compute_free_energy,
    compute_t_score,
    compute_variance,
)

## 1. Build the system

The `"triangular strip open"` lattice and `"chiral spin"` model are
included in QALMA's built-in library (see `qalma/lib/lattices.xml` and
`qalma/lib/models.xml`).  Parameter `Wilson2` sets the chirality
coupling; `J` sets the Heisenberg coupling on all bonds.

In [ ]:
L   = 6      # number of unit cells  →  2*L sites
J   = 1.0    # Heisenberg NN coupling
chi = 0.5    # scalar chirality coupling
beta = 2.0   # inverse temperature

graph  = graph_from_alps_xml(name='triangular strip open',
                             parms={'L': L, 'a': 1})
model  = model_from_alps_xml(name='chiral spin')
system = SystemDescriptor(graph, model, {'J': J, 'Wilson2': chi})
ham    = system.global_operator('Hamiltonian')

sites = tuple(sorted(system.sites.keys()))
N     = len(sites)
print(f'Sites ({N} total): {sites}')
print(f'Loops: {graph.loops}')

## 2. Exact free energy (small system)

For $L\le 8$ we can diagonalise exactly.  The exact free energy is
$F_{\rm exact} = -\log Z = -\log\mathrm{Tr}[e^{-\beta H}]$, in the
same units as $\beta H$.

> **Note**: use a numerically stable shift by the ground-state energy
> $E_0$ to avoid overflow:
> $\log Z_{\rm shift} = \log\sum_i e^{-\beta(E_i-E_0)} = \log Z + \beta E_0$,
> so $-\log Z = -\log Z_{\rm shift} + \beta E_0$.

In [ ]:
def exact_free_energy(ham, sites, beta):
    evals = ham.to_qutip(sites).eigenenergies()
    e0 = evals.min()
    log_Z_shift = np.log(np.exp(-beta * (evals - e0)).sum())
    return -log_Z_shift + beta * e0

f_exact = exact_free_energy(ham, sites, beta)
print(f'F_exact = {f_exact:.6f}')

## 3. Self-consistent baseline

The self-consistent MF solution (`numfields=0`) is the starting point.
It minimises $F[\sigma]$ within the space of product states whose
generator is the mean-field of the original Hamiltonian.

In [ ]:
sigma_sc = variational_quadratic_mfa(
    beta * ham, numfields=0, max_self_consistent_steps=100
)
f_sc   = compute_free_energy(sigma_sc,  beta * ham)
var_sc = compute_variance(sigma_sc, beta * ham)
ts_sc, _, _ = compute_t_score(sigma_sc, beta * ham, f_exact)

print(f'SC baseline:')
print(f'  F       = {f_sc:.6f}  (F_exact = {f_exact:.6f})')
print(f'  Var(F^) = {var_sc:.4g}')
print(f'  T-score = {ts_sc:.4f}')

## 4. Variational sweep over number of fields

We sweep `numfields` from 1 to 10, using each solution as a warm start
for the next.  For each $m$ we record:

- $F[\sigma_m]$ — variational free energy
- $T_{\rm score}[\sigma_m]$ — needs $F_{\rm exact}$, only for small systems
- $R_m = \mathrm{Var}[\sigma_m]/\mathrm{Var}[\sigma_{\rm SC}]$ — always available

In [ ]:
numfields_list = [1, 2, 3, 4, 6, 8, 10]

records = []
sigma_ref = sigma_sc

for nf in numfields_list:
    sigma_ref = variational_quadratic_mfa(
        beta * ham,
        numfields=nf,
        sigma_ref=sigma_ref,
        max_self_consistent_steps=30,
    )
    f   = compute_free_energy(sigma_ref, beta * ham)
    var = compute_variance(sigma_ref, beta * ham)
    ts, _, _ = compute_t_score(sigma_ref, beta * ham, f_exact)
    R   = var / var_sc
    records.append({'nf': nf, 'f': f, 'var': var, 'R': R, 'tscore': ts})
    print(f'nf={nf:2d}:  F={f:.6f}  T-score={ts:.4f}  R={R:.4f}')

## 5. Plots

Three panels:

- **(a)** Variational free energy $F[\sigma_m]$ vs $m$, with $F_{\rm exact}$ as reference.
- **(b)** T-score vs $m$ (log scale) — measures residual fluctuations
  relative to the gap $\langle\hat{F}\rangle$.
- **(c)** Variance ratio $R_m$ vs $m$ — reduction in fluctuations
  relative to the SC baseline; does not require $F_{\rm exact}$.

In [ ]:
fig = plt.figure(figsize=(11, 3.5))
gs  = gridspec.GridSpec(1, 3, figure=fig, wspace=0.38)
axes = [fig.add_subplot(gs[i]) for i in range(3)]

nfs    = [r['nf']     for r in records]
fs     = [r['f']      for r in records]
ts_arr = [r['tscore'] for r in records]
Rs     = [r['R']      for r in records]

# --- (a) Free energy ---
ax = axes[0]
ax.plot(nfs, fs, 'o-', color='#1F77B4', label='Variational MF')
ax.axhline(f_sc,    color='#E87833', linestyle='--', label='SC baseline')
ax.axhline(f_exact, color='0.4',     linestyle=':',  label='Exact')
ax.set_xlabel('Number of fields $m$')
ax.set_ylabel(r'$F[\sigma_m]$')
ax.set_title('Free energy')
ax.legend(fontsize=8)
ax.text(-0.18, 1.02, '(a)', transform=ax.transAxes, fontweight='bold')

# --- (b) T-score ---
ax = axes[1]
ax.plot(nfs, ts_arr, 'o-', color='#1F77B4')
ax.axhline(ts_sc, color='#E87833', linestyle='--', label='SC')
ax.set_xlabel('Number of fields $m$')
ax.set_ylabel(r'$T_{\rm score}$')
ax.set_title('T-score')
ax.set_yscale('log')
ax.legend(fontsize=8)
ax.text(-0.18, 1.02, '(b)', transform=ax.transAxes, fontweight='bold')

# --- (c) Variance ratio ---
ax = axes[2]
ax.plot(nfs, Rs, 'o-', color='#1F77B4')
ax.axhline(1.0, color='#E87833', linestyle='--', label='SC baseline')
ax.set_xlabel('Number of fields $m$')
ax.set_ylabel(r'$R_m = \mathrm{Var}[\sigma_m]\,/\,\mathrm{Var}[\sigma_{\rm SC}]$')
ax.set_title('Variance ratio')
ax.set_ylim(-0.05, 1.15)
ax.legend(fontsize=8)
ax.text(-0.18, 1.02, '(c)', transform=ax.transAxes, fontweight='bold')

fig.suptitle(
    f'Chiral strip  $L={L}$,  $J={J}$,  $K={chi}$,  $\\beta={beta}$',
    y=1.02
)
plt.savefig('chiral_variational_mf.pdf', bbox_inches='tight')
plt.show()

## 6. Effect of frustration: chirality coupling sweep

We now fix $m=6$ fields and vary $K/J$ from 0 to 2 to see how chirality
affects the quality of the mean-field approximation.

In [ ]:
chi_vals = np.linspace(0, 2.0, 9)
nf_fixed = 6

results_chi = []
for chi_v in chi_vals:
    system_v = SystemDescriptor(
        graph_from_alps_xml(name='triangular strip open', parms={'L': L, 'a': 1}),
        model_from_alps_xml(name='chiral spin'),
        {'J': J, 'Wilson2': chi_v},
    )
    ham_v = system_v.global_operator('Hamiltonian')
    sites_v = tuple(sorted(system_v.sites.keys()))

    sigma_sc_v = variational_quadratic_mfa(
        beta * ham_v, numfields=0, max_self_consistent_steps=100
    )
    sigma_var_v = variational_quadratic_mfa(
        beta * ham_v, numfields=nf_fixed, sigma_ref=sigma_sc_v,
        max_self_consistent_steps=30,
    )
    f_ex  = exact_free_energy(ham_v, sites_v, beta)
    f_v   = compute_free_energy(sigma_var_v, beta * ham_v)
    var_sc_v  = compute_variance(sigma_sc_v,  beta * ham_v)
    var_var_v = compute_variance(sigma_var_v, beta * ham_v)
    ts_v, _, _ = compute_t_score(sigma_var_v, beta * ham_v, f_ex)
    R_v = var_var_v / var_sc_v if var_sc_v > 1e-15 else float('nan')
    results_chi.append({'chi': chi_v, 'f': f_v, 'f_exact': f_ex,
                        'tscore': ts_v, 'R': R_v})

fig2, axes2 = plt.subplots(1, 3, figsize=(11, 3.5))
chis   = [r['chi']    for r in results_chi]
dFs    = [r['f'] - r['f_exact'] for r in results_chi]
tscores = [r['tscore'] for r in results_chi]
Rs2    = [r['R']      for r in results_chi]

axes2[0].plot(chis, dFs, 'o-', color='#1F77B4')
axes2[0].set_xlabel('$K/J$'); axes2[0].set_ylabel(r'$F[\sigma] - F_{\rm exact}$')
axes2[0].set_title('Free energy gap')
axes2[0].text(-0.18, 1.02, '(a)', transform=axes2[0].transAxes, fontweight='bold')

axes2[1].plot(chis, tscores, 'o-', color='#1F77B4')
axes2[1].set_xlabel('$K/J$'); axes2[1].set_ylabel(r'$T_{\rm score}$')
axes2[1].set_title('T-score')
axes2[1].set_yscale('log')
axes2[1].text(-0.18, 1.02, '(b)', transform=axes2[1].transAxes, fontweight='bold')

axes2[2].plot(chis, Rs2, 'o-', color='#1F77B4')
axes2[2].axhline(1.0, color='#E87833', linestyle='--')
axes2[2].set_xlabel('$K/J$'); axes2[2].set_ylabel('$R_m$')
axes2[2].set_title('Variance ratio')
axes2[2].set_ylim(-0.05, 1.15)
axes2[2].text(-0.18, 1.02, '(c)', transform=axes2[2].transAxes, fontweight='bold')

fig2.suptitle(
    f'Effect of chirality coupling  ($L={L}$, $m={nf_fixed}$, $\\beta={beta}$)',
    y=1.02
)
plt.savefig('chiral_coupling_sweep.pdf', bbox_inches='tight')
plt.show()

## Summary

- The variational free energy $F[\sigma_m]$ decreases monotonically with $m$
  and converges to $F_{\rm exact}$ from above.
- The **T-score** captures a complementary aspect: even when $F[\sigma_m]$
  is close to $F_{\rm exact}$, a large T-score signals that the fluctuation
  structure of the approximation is still far from the exact Gibbs state.
- The **variance ratio** $R_m$ provides the same qualitative information
  without requiring $F_{\rm exact}$, making it suitable for large systems.
- For the chiral model, increasing $K/J$ generally makes the mean-field
  approximation harder (larger T-score), reflecting the frustration
  introduced by the three-body term.

### References

- User guide: :doc:`../user/loop_operators`
- User guide: :doc:`../user/meanfield_variational`
- API: :func:`qalma.meanfield.variational_quadratic_mfa`
- API: :func:`qalma.meanfield.compute_t_score`
- API: :func:`qalma.meanfield.compute_variance`